# Relation Extraction — Full Corpus

This notebook implements the relation extraction (RE) pipeline for building a clinical oncology knowledge graph from NCCN guidelines.

## Pipeline Overview

The pipeline uses two complementary LLM passes over pre-chunked guideline text:

| Pass | Model | Predicate Group | Predicates |
|------|-------|----------------|------------|
| Core | DeepSeek-V3 | Direct clinical relations | TREATS, COMBINED_WITH, REQUIRES_BIOMARKER, HAS_DOSE, HAS_FREQUENCY, HAS_ROUTE, HAS_EVIDENCE_LEVEL, HAS_STRENGTH, ALTERNATIVE_TO, CONTRAINDICATED_IN |
| Context | GPT-4.1 | Clinical context attributes | INDICATED_FOR_STAGE, USED_IN_POPULATION, ACHIEVES_OUTCOME, HAS_ELIGIBILITY |

**Input:** `chunk_index.json` (chunked text) + `normalized_entities.json` (GLiNER + BEL output)  
**Output:** `re_merged_final.json` — deduplicated triple store ready for Neo4j import

## Requirements
```
pip install openai json-repair tqdm
```
API keys required: `DEEPSEEK_API_KEY`, `OPENAI_API_KEY`


In [ ]:
# ── Cell 1: Dependencies ─────────────────────────────────────────────────────
!pip install openai json-repair tqdm -q

import os, json, time, re
from pathlib import Path
from collections import Counter
from openai import OpenAI
from json_repair import repair_json


In [ ]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────

DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY', '<your-deepseek-api-key>')
OPENAI_API_KEY   = os.environ.get('OPENAI_API_KEY',   '<your-openai-api-key>')

DATA_DIR     = Path('data')
RESULTS_DIR  = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

DS_CHECKPOINT  = RESULTS_DIR / 're_deepseek_checkpoint.json'
GPT_CHECKPOINT = RESULTS_DIR / 're_gpt41_checkpoint.json'
DS_FINAL       = RESULTS_DIR / 're_deepseek_final.json'
GPT_FINAL      = RESULTS_DIR / 're_gpt41_final.json'
MERGED_FINAL   = RESULTS_DIR / 're_merged_final.json'

CHUNK_INDEX_PATH = DATA_DIR / 'chunk_index.json'
ENTITIES_PATH    = DATA_DIR / 'normalized_entities.json'

CHECKPOINT_EVERY = 100

CORE_PREDICATES = {
    'TREATS', 'COMBINED_WITH', 'REQUIRES_BIOMARKER', 'HAS_DOSE',
    'HAS_FREQUENCY', 'HAS_ROUTE', 'HAS_EVIDENCE_LEVEL', 'HAS_STRENGTH',
    'ALTERNATIVE_TO', 'CONTRAINDICATED_IN'
}
CONTEXT_PREDICATES = {
    'INDICATED_FOR_STAGE', 'USED_IN_POPULATION', 'ACHIEVES_OUTCOME', 'HAS_ELIGIBILITY'
}


In [ ]:
# ── Cell 3: Load Data ────────────────────────────────────────────────────────

with open(CHUNK_INDEX_PATH) as f:
    chunk_index = json.load(f)

with open(ENTITIES_PATH) as f:
    entities_index = json.load(f)

all_chunks = [(doc_id, chunk_num)
              for doc_id, chunks in chunk_index.items()
              for chunk_num in chunks]

def get_chunk_text(doc_id, chunk_num):
    return chunk_index.get(doc_id, {}).get(str(chunk_num), {}).get('text', '')

def get_chunk_entities(doc_id, chunk_num):
    return entities_index.get(doc_id, {}).get(str(chunk_num), [])

print(f'Documents loaded: {len(chunk_index):,}')
print(f'Total chunks: {len(all_chunks):,}')


In [ ]:
# ── Cell 4: Prompts and Normalization ────────────────────────────────────────

SYSTEM_PROMPT_CORE = """You are a clinical NLP expert specializing in oncology.
Extract ONLY direct clinical relationships from the provided NCCN guideline text.
Focus EXCLUSIVELY on these relation types:
  TREATS                (drug/therapy treats a disease or condition)
  CONTRAINDICATED_IN    (drug/therapy is contraindicated in a condition)
  REQUIRES_BIOMARKER    (treatment requires a specific biomarker/mutation)
  COMBINED_WITH         (drug used in combination with another drug/therapy)
  HAS_DOSE              (drug given at a specific dose, e.g. "500 mg")
  HAS_FREQUENCY         (drug given at a specific frequency, e.g. "every 3 weeks")
  HAS_ROUTE             (drug given via a specific route, e.g. "IV", "oral")
  HAS_EVIDENCE_LEVEL    (treatment has NCCN category of evidence, e.g. "Category 1")
  HAS_STRENGTH          (treatment strength: "preferred", "useful in certain circumstances")
  ALTERNATIVE_TO        (treatment is an alternative to another treatment)

Guidelines:
1. Extract ONLY from the provided text. Do not invent relations.
2. Provide the EXACT text fragment as \"evidence\".
3. Return ONLY a valid JSON array with keys: subject, predicate, object, confidence, evidence.
4. Set confidence to \"HIGH\" if explicitly stated, \"LOW\" if implied.
5. DO NOT extract INDICATED_FOR_STAGE, USED_IN_POPULATION, ACHIEVES_OUTCOME, HAS_ELIGIBILITY.
"""

# Context-pass prompt (GPT-4.1) targets 4 additional predicate types:
# INDICATED_FOR_STAGE, USED_IN_POPULATION, ACHIEVES_OUTCOME, HAS_ELIGIBILITY
# See README for full prompt details.
SYSTEM_PROMPT_CONTEXT = None  # define your context prompt here

def normalize_object(val: str, predicate: str) -> str:
    """Normalize object values to canonical forms (domain-specific)."""
    return val.strip().lower()

def clean_triples(raw, doc_id, chunk_num, method, allowed_predicates=None):
    results = []
    for t in raw:
        if not isinstance(t, dict):
            continue
        pred = str(t.get('predicate', '')).strip().upper()
        if allowed_predicates and pred not in allowed_predicates:
            continue
        subj = str(t.get('subject', '')).strip().lower()
        obj  = normalize_object(str(t.get('object', '')).strip(), pred)
        conf_raw = str(t.get('confidence', 'LOW')).strip().upper()
        conf = 1.0 if conf_raw == 'HIGH' else 0.5
        if not subj or not obj:
            continue
        results.append({
            'subject':         subj,
            'predicate':       pred,
            'object':          obj,
            'object_original': str(t.get('object', '')).strip(),
            'confidence':      conf,
            'evidence':        str(t.get('evidence', '')).strip()[:300],
            'method':          method,
            'doc_id':          doc_id,
            'chunk_id':        chunk_num,
        })
    return results

def dedup_triples(triples):
    seen = {}
    for t in triples:
        key = (t['subject'], t['predicate'], t['object'], t['doc_id'])
        if key not in seen or t['confidence'] > seen[key]['confidence']:
            seen[key] = t
    return list(seen.values())


print(f'Core predicates ({len(CORE_PREDICATES)}): {sorted(CORE_PREDICATES)}')
print(f'Context predicates ({len(CONTEXT_PREDICATES)}): {sorted(CONTEXT_PREDICATES)}')


In [ ]:
# ── Cell 5: API Clients ──────────────────────────────────────────────────────

client_ds     = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com')
client_openai = OpenAI(api_key=OPENAI_API_KEY)

def call_with_backoff(client, model, system_prompt, user_prompt, max_retries=5):
    """API call with exponential backoff on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt}
                ],
                temperature=0.0,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = 2 ** attempt * 5
                print(f'  [rate-limit] waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                print(f'  [ERR] {err[:120]}')
                if attempt == max_retries - 1:
                    return None
                time.sleep(2)
    return None



In [ ]:
# ── Cell 6: Extraction Functions ─────────────────────────────────────────────

def build_user_prompt_core(chunk_text, entities, max_entities=30):
    entity_list, seen = [], set()
    for e in entities:
        key = e['text'].strip().lower()
        if key not in seen:
            seen.add(key)
            entity_list.append(f"- {e['text']} [{e['label']}]")
        if len(entity_list) >= max_entities:
            break
    ents_str = '\n'.join(entity_list) if entity_list else '(no pre-identified entities)'
    return f"""ENTITIES IN THIS TEXT:
{ents_str}

TEXT:
{chunk_text[:5500]}

Extract ALL core relations between the listed entities.
Return JSON array only:"""

def build_user_prompt_context(chunk_text):
    return f"""TEXT:
{chunk_text[:5500]}

Find ALL instances of INDICATED_FOR_STAGE, USED_IN_POPULATION, ACHIEVES_OUTCOME, HAS_ELIGIBILITY.
Return JSON array only:"""

def extract_core(doc_id, chunk_num, chunk_text, entities):
    if not chunk_text.strip():
        return []
    raw = call_with_backoff(client_ds, 'deepseek-chat', SYSTEM_PROMPT_CORE,
                            build_user_prompt_core(chunk_text, entities))
    if raw is None:
        return []
    try:
        return clean_triples(json.loads(repair_json(raw)), doc_id, chunk_num,
                             'deepseek_v3_core', allowed_predicates=CORE_PREDICATES)
    except json.JSONDecodeError:
        return []

def extract_context(doc_id, chunk_num, chunk_text):
    if not chunk_text.strip() or SYSTEM_PROMPT_CONTEXT is None:
        return []
    raw = call_with_backoff(client_openai, 'gpt-4.1', SYSTEM_PROMPT_CONTEXT,
                            build_user_prompt_context(chunk_text))
    if raw is None:
        return []
    try:
        return clean_triples(json.loads(repair_json(raw)), doc_id, chunk_num,
                             'gpt41_context', allowed_predicates=CONTEXT_PREDICATES)
    except json.JSONDecodeError:
        return []


In [ ]:
# ── Cell 7: Checkpoint Helpers ───────────────────────────────────────────────

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'Checkpoint loaded: {len(data["results"]):,} triples, '
              f'{data["processed_count"]} chunks processed')
        return data
    return {'results': [], 'processed_chunks': [], 'processed_count': 0}

def save_checkpoint(path, results, processed_chunks, processed_count):
    with open(path, 'w') as f:
        json.dump({'results': results,
                   'processed_chunks': list(processed_chunks),
                   'processed_count': processed_count}, f, ensure_ascii=False)



In [ ]:
# ── Cell 8: Run DeepSeek-V3 — Core Predicates ───────────────────────────────

cp = load_checkpoint(DS_CHECKPOINT)
ds_results         = cp['results']
processed_ds       = set(tuple(x) for x in cp.get('processed_chunks', []))
processed_count_ds = cp.get('processed_count', 0)

remaining = [(d, c) for d, c in all_chunks if (d, str(c)) not in processed_ds]
print(f'DeepSeek-V3 Core RE | chunks remaining: {len(remaining):,}\n')

start = time.time()
for i, (doc_id, chunk_num) in enumerate(remaining):
    triples = extract_core(doc_id, chunk_num,
                           get_chunk_text(doc_id, chunk_num),
                           get_chunk_entities(doc_id, chunk_num))
    ds_results.extend(triples)
    processed_ds.add((doc_id, str(chunk_num)))
    processed_count_ds += 1

    if (i + 1) % 50 == 0:
        rate    = (i + 1) / (time.time() - start)
        eta_min = int(len(remaining[i+1:]) / rate / 60) if rate > 0 else 0
        print(f'  [{processed_count_ds:,}/{len(all_chunks):,}] '
              f'{len(ds_results):,} triples | {rate:.1f} chunks/s | ETA ~{eta_min} min')

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(DS_CHECKPOINT, ds_results, processed_ds, processed_count_ds)

    time.sleep(1)  # respect API rate limits

ds_final = dedup_triples(ds_results)
with open(DS_FINAL, 'w') as f:
    json.dump(ds_final, f, ensure_ascii=False, indent=2)

print(f'   Triples (before dedup): {len(ds_results):,}')
print(f'   Triples (after dedup):  {len(ds_final):,}')


In [ ]:
# ── Cell 9: Run GPT-4.1 — Context Predicates ────────────────────────────────
# Run after Cell 8 (or in parallel in a separate session)
# Requires SYSTEM_PROMPT_CONTEXT to be defined in Cell 4

cp_gpt = load_checkpoint(GPT_CHECKPOINT)
gpt_results         = cp_gpt['results']
processed_gpt       = set(tuple(x) for x in cp_gpt.get('processed_chunks', []))
processed_count_gpt = cp_gpt.get('processed_count', 0)

remaining_gpt = [(d, c) for d, c in all_chunks if (d, str(c)) not in processed_gpt]
print(f'GPT-4.1 Context RE | chunks remaining: {len(remaining_gpt):,}\n')

start_gpt = time.time()
for i, (doc_id, chunk_num) in enumerate(remaining_gpt):
    triples = extract_context(doc_id, chunk_num, get_chunk_text(doc_id, chunk_num))
    gpt_results.extend(triples)
    processed_gpt.add((doc_id, str(chunk_num)))
    processed_count_gpt += 1

    if (i + 1) % 50 == 0:
        rate    = (i + 1) / (time.time() - start_gpt)
        eta_min = int(len(remaining_gpt[i+1:]) / rate / 60) if rate > 0 else 0
        print(f'  [{processed_count_gpt:,}/{len(all_chunks):,}] '
              f'{len(gpt_results):,} triples | {rate:.1f} chunks/s | ETA ~{eta_min} min')

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(GPT_CHECKPOINT, gpt_results, processed_gpt, processed_count_gpt)

    time.sleep(1)

gpt_final = dedup_triples(gpt_results)
with open(GPT_FINAL, 'w') as f:
    json.dump(gpt_final, f, ensure_ascii=False, indent=2)

print(f'   Triples (before dedup): {len(gpt_results):,}')
print(f'   Triples (after dedup):  {len(gpt_final):,}')


In [ ]:
# ── Cell 10: Merge Core + Context ───────────────────────────────────────────

with open(DS_FINAL) as f:
    ds_final = json.load(f)
print(f'DeepSeek core:   {len(ds_final):,} triples')

if GPT_FINAL.exists():
    with open(GPT_FINAL) as f:
        gpt_final = json.load(f)
    print(f'GPT-4.1 context: {len(gpt_final):,} triples')
    merged_raw = ds_final + gpt_final
else:
    print('GPT-4.1 output not found — using core pass only')
    merged_raw = ds_final

merged = dedup_triples(merged_raw)
print(f'\nMerged (before dedup): {len(merged_raw):,}')
print(f'Merged (after dedup):  {len(merged):,}')

pred_dist = Counter(t.get('predicate', '') for t in merged)
print('\n── Predicate distribution ──')
for pred, cnt in sorted(pred_dist.items(), key=lambda x: -x[1]):
    layer = 'core' if pred in CORE_PREDICATES else 'ctx'
    print(f'  [{layer}] {pred:30s}: {cnt:,}')

doc_dist = Counter(t.get('doc_id', '') for t in merged)
print('\n── Top-10 documents by triple count ──')
for doc, cnt in doc_dist.most_common(10):
    print(f'  {doc}: {cnt:,}')

with open(MERGED_FINAL, 'w') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)


print(f'   Total triples: {len(merged):,}')
